In [0]:
dim_date = spark.read.table(
    "workspace.gold.dim_date"
)

dim_location = spark.read.table(
    "workspace.gold.dim_location"
)

dim_magnitude = spark.read.table(
    "workspace.gold.dim_magnitude"
)

dim_status = spark.read.table(
    "workspace.gold.dim_status"
)


In [0]:
silver_df = spark.read.table(
    "workspace.silver.earthquakes"
)
#display(silver_df)

In [0]:
from pyspark.sql.functions import col, when

fact_df = (
    silver_df
    .withColumn(
        "magnitude_category",
        when(col("magnitude") < 4, "Bajo")
        .when(col("magnitude") < 5, "Moderado")
        .when(col("magnitude") < 6, "Fuerte")
        .otherwise("Severo")
    )
)

In [0]:
from pyspark.sql.functions import date_format

fact_df = fact_df.join(
    dim_date,
    date_format(
        fact_df.event_datetime,
        "yyyyMMdd"
    ).cast("int") == dim_date.date_key,
    "left"
)

In [0]:
fact_df = fact_df.join(
    dim_location.select(
        "location_key",
        "latitude",
        "longitude"
    ),
    [
        fact_df.latitude == dim_location.latitude,
        fact_df.longitude == dim_location.longitude
    ],
    "left"
)

In [0]:
fact_df = fact_df.join(
    dim_magnitude,
    fact_df.magnitude_category == dim_magnitude.category,
    "left"
)

In [0]:
fact_df = fact_df.join(
    dim_status,
    fact_df.status == dim_status.status,
    "left"
)

In [0]:
fact_earthquakes = fact_df.select(
    "event_id",
    "title",
    "date_key",
    "location_key",
    "magnitude_key",
    "status_key",
    "magnitude",
    "km",
    "significance",
    "tsunami"
)
#display(fact_earthquakes)

In [0]:
print("Silver:", silver_df.count())
print("Fact:", fact_earthquakes.count())

In [0]:
#%sql
#DROP TABLE IF EXISTS workspace.gold.fact_earthquakes;

In [0]:
(
    fact_earthquakes
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.fact_earthquakes"
    )
)

print("Fact creada correctamente")